# GISPR Module 3 — Vector Data Intake

**Course:** GIS Spatial Analysis with Python and R (GISPR)
**Module:** 3 — Vector Data Intake: Loading, CRS, Cleaning, Harmonizing
**Builds on:** Module 2 (loading spatial data, loops, functions, Git basics)
**Feeds into:** Module 4 (spatial operations — buffer, clip, overlay, spatial join run on the data you clean here)

---

### Tonight's framing problem

You receive a CSV of salmon observations. The watershed boundaries your agency needs to join them against live in a File Geodatabase. A colleague also points you at a hosted feature layer on ArcGIS Online with an updated watershed layer, and a partner agency's GeoJSON export. Four different sources, one goal: a single harmonized GeoPackage that opens cleanly in ArcGIS Pro, R, and Python.

That's the intake routine this module builds, step by step, in both languages.

### Tonight's Learning Objectives

| # | Learning outcome | Notebook section |
|---|---|---|
| 1 | Load vector data from any source — file geodatabase (arcpy), hosted feature layer (ArcGIS API for Python / SeDF), file formats (GeoPandas / sf), and R via the R-ArcGIS Bridge | Section 1 |
| 2 | Identify the CRS of any spatial dataset and reproject to a common target CRS in both Python and R | Section 2 |
| 3 | Clean and standardize attribute tables — column names, data types, null drops | Section 3 |
| 4 | Convert tabular data with coordinate columns into spatial objects, and confirm geometry validity with a plot | Section 4 |
| 5 | Write harmonized vector data to a single GeoPackage readable by ArcGIS Pro, R, and Python | Section 5 |

### Kernel reminder
- 🔵 **`[R]`** cells — switch kernel to **R** before running
- 🟢 **`[Python]`** cells — switch kernel to **Python 3** (or your cloned ArcGIS Pro env) before running
- 📋 **`[Terminal]`** cells — paste the command into your terminal / Anaconda Prompt, **not** in a notebook cell

### A note on coordinate reference systems in this notebook

Every reprojection step below targets **EPSG:32610 (UTM Zone 10N)** — the course-wide standard CRS for Pacific Northwest analysis, used for distance and area measurement in meters. If your live-session slide deck shows the Guided Lab targeting EPSG:2926 (WA State Plane North, feet) instead, flag it — that slide and the Extended Application "Build a Reusable Function" challenge (which targets EPSG:32610) are inconsistent with each other, and 32610 is the standard used everywhere else in the course materials.


---

## Section 1 — Loading Vector Data from Any Source

Tonight's data almost never starts in one tidy format. It might live in a File Geodatabase your agency maintains, a hosted feature layer on ArcGIS Online, a GeoJSON a partner agency emailed you, or a CSV with lat/lon columns. The good news: once loaded, all four sources end up in the same kind of object — a table with a geometry column — and the same four-step intake routine applies regardless of where the data came from.

### Four sources, one intake routine

| Source | Tool | Returns |
|---|---|---|
| File Geodatabase (`.gdb`) | `arcpy.da.SearchCursor` / `GeoAccessor.from_featureclass()` | Cursor rows / Spatially Enabled DataFrame |
| Hosted Feature Layer (AGOL/Enterprise) | ArcGIS API for Python (`FeatureLayer.query()`) | Spatially Enabled DataFrame (SeDF) |
| File formats (Shapefile, GeoJSON, GeoPackage) | `geopandas.read_file()` / `sf::st_read()` | GeoDataFrame / sf object |
| Geodatabase from R | R-ArcGIS Bridge (`arcgisbinding`) | sf object via `arc.data2sf()` |

> **Why does this matter to me?** Production GIS data rarely lives in one place. Knowing how to bring all four source types into a common in-memory structure means you can write one cleaning/export pipeline instead of four.

### Three ways to hold spatial data in code

| | sf (R) | GeoPandas (Python) | SeDF (Python/ESRI) |
|---|---|---|---|
| Standard for | R | Open-source Python | ESRI ecosystem |
| Structure | data.frame + list-column geometry (`sfc`) | pandas DataFrame + `geometry` column (Shapely) | pandas DataFrame + `SHAPE` column |
| Best for | Tidyverse workflows, `dplyr` verbs work natively | File-based, open-source workflows | Direct ArcGIS Online / Portal access |

Key insight: all three implement the same concept — a table where one column holds geometry. The difference is ecosystem and interop, not the underlying idea. `GeoAccessor.from_geodataframe()` and `sdf.spatial.to_geodataframe()` convert between SeDF and GeoPandas when you need to cross ecosystems.


### 1a — Loading from a File Geodatabase (`arcpy` and the ArcGIS API for Python)

In [ ]:
# [Python / ArcPy] Reading vector data from a File Geodatabase
# Run this inside ArcGIS Pro's built-in Notebook, or in a cloned arcgispro-py3 environment.
# This cell is illustrative — uncomment everything (including the import) to run it live.

# import arcpy
import os

# ── Set workspace to your File GDB ───────────────────────────────────────────
# arcpy.env.workspace = r'C:\Projects\WA_GIS\data\salmon_watersheds.gdb'

# ── List all feature classes in the GDB ─────────────────────────────────────
# fcs = arcpy.ListFeatureClasses()
# print("Feature classes:", fcs)

# ── Describe a feature class — geometry type, CRS, field info ────────────────
# desc = arcpy.Describe('watersheds')
# print("Geometry type:", desc.shapeType)
# print("Spatial reference:", desc.spatialReference.name)
# print("WKID:", desc.spatialReference.factoryCode)

# ── Option 1: row-by-row read with SearchCursor ───────────────────────────────
# with arcpy.da.SearchCursor('watersheds', ["WATERSHED_ID", "WATERSHED_NAME"]) as cur:
#     rows = [row for row in cur]
# print(f"Read {len(rows)} rows via SearchCursor")

# ── Option 2: read directly into a Spatially Enabled DataFrame ───────────────
# from arcgis.features import GeoAccessor
# watersheds_sdf = pd.DataFrame.spatial.from_featureclass('watersheds')
# print(watersheds_sdf.spatial.sr)     # CRS, e.g. {'wkid': 2926}
# print(watersheds_sdf.shape)

print("ArcPy / GDB cell — uncomment and point at a real .gdb path to run live.")
print("If arcpy is unavailable in your environment, use the GeoPandas/sf cells in Section 1c instead.")


### 1b — Loading from a hosted feature layer (ArcGIS API for Python)

In [ ]:
# [Python] ArcGIS API for Python — accessing vector data from ArcGIS Online or Portal
# Requires: pip install arcgis (or included in the ArcGIS Pro Python environment)

# from arcgis.gis import GIS
# from arcgis.features import FeatureLayer
# import geopandas as gpd

# ── Connect (anonymous works for public data; pass credentials for Portal) ───
# gis = GIS()   # anonymous public access
# gis = GIS('home')  # signed-in session inside ArcGIS Pro

# ── Search for a public feature layer ────────────────────────────────────────
# results = gis.content.search('salmon observations watershed', item_type='Feature Layer')
# for item in results[:3]:
#     print(item.title, '|', item.url)

# ── Access a feature layer by URL and query to a Spatially Enabled DataFrame ──
# layer_url = 'https://services.arcgis.com/.../FeatureServer/0'
# flayer = FeatureLayer(layer_url)
# sdf = flayer.query().sdf
# print(type(sdf))
# print(sdf.spatial.sr)            # CRS dict, e.g. {'wkid': 4326}

# ── Filtered + reprojected query in one call ──────────────────────────────────
# sdf_filtered = flayer.query(where="species='Chinook'", out_sr=32610).sdf
# print(sdf_filtered.shape)

# ── Convert SeDF to GeoPandas for open-source analysis downstream ────────────
# watersheds_gdf = sdf.spatial.to_geodataframe()

print("ArcGIS API cell — uncomment and run with a valid feature layer URL or AGOL search.")
print("See: https://developers.arcgis.com/python/ for full API docs.")


### 1c — Loading file formats in Python (`geopandas`) and R (`sf`)

In [ ]:
# [Python] Reading vector formats with geopandas — Shapefile, GeoJSON, GeoPackage
import geopandas as gpd

# ── Example: read a salmon observation watershed GeoPackage (multi-layer) ────
# List the layers first if you don't already know the layer names:
# import fiona
# print(fiona.listlayers('data/salmon_watersheds.gpkg'))

# watersheds = gpd.read_file('data/salmon_watersheds.gpkg', layer='watersheds')
# print("Geometry type:", watersheds.geom_type.unique())
# print("Feature count:", len(watersheds))
# print("CRS:", watersheds.crs)

# ── Example: read a GeoJSON from a partner agency ─────────────────────────────
# obs_geojson = gpd.read_file('data/salmon_obs_partner_agency.geojson')
# print(obs_geojson.crs)   # GeoJSON is WGS84 (EPSG:4326) by convention

# ── Example: read a Shapefile ──────────────────────────────────────────────────
# parcels = gpd.read_file('data/parcels.shp')

print("GeoPandas read_file cell — uncomment and point at your own file paths to run live.")


In [ ]:
# [R] Reading vector formats with sf — Shapefile, GeoJSON, GeoPackage
library(sf)

# ── Example: list layers in a multi-layer GeoPackage, then read one ──────────
# st_layers('data/salmon_watersheds.gpkg')
# watersheds <- st_read('data/salmon_watersheds.gpkg', layer = 'watersheds', quiet = TRUE)
# print(st_geometry_type(watersheds)[1])
# print(nrow(watersheds))
# print(st_crs(watersheds)$epsg)

# ── Example: read a GeoJSON from a partner agency ─────────────────────────────
# obs_geojson <- st_read('data/salmon_obs_partner_agency.geojson', quiet = TRUE)
# st_crs(obs_geojson)$epsg   # GeoJSON is WGS84 (EPSG:4326) by convention

cat("sf::st_read cell — uncomment and point at your own file paths to run live.\n")


### 1d — Loading from a geodatabase in R (R-ArcGIS Bridge)

In [ ]:
# [R] R-ArcGIS Bridge — read a File Geodatabase feature class from R
# Requires: ArcGIS Pro installed + the arcgisbinding package configured

# library(arcgisbinding)
# arc.check_product()                 # confirms the Bridge can see your ArcGIS Pro license

# ── Open and select ────────────────────────────────────────────────────────
# arc_obj  <- arc.open("salmon_watersheds.gdb/watersheds")
# arc_data <- arc.select(arc_obj, fields = c("WATERSHED_ID", "WATERSHED_NAME"))

# ── Convert to sf for downstream analysis ─────────────────────────────────────
# watersheds_sf <- arc.data2sf(arc_data)
# st_crs(watersheds_sf)$epsg
# nrow(watersheds_sf)
# st_geometry_type(watersheds_sf)[1]

# ── Write results back to the geodatabase ─────────────────────────────────────
# arc.write("salmon_watersheds.gdb/watersheds_cleaned", watersheds_sf)

cat("R-ArcGIS Bridge cell — uncomment and run inside an environment with ArcGIS Pro + the Bridge installed.\n")
cat("No ArcGIS Pro license? The arcgislayers package reads hosted feature services over REST instead:\n")
cat('  library(arcgislayers); arc_read(url) -> sf object. No Pro license required.\n')


---

## Section 2 — CRS Check and Reprojection

CRS errors are the #1 source of silent failures in spatial analysis. A layer that plots in the wrong place, measurements in degrees instead of meters, or an overlay that's thousands of kilometers off — all CRS problems.

### The four CRS questions to ask every time you load data

1. **What CRS is it?** — `gdf.crs` / `st_crs(sf_obj)`
2. **Is it geographic or projected?** — degrees (geographic) vs. meters/feet (projected)
3. **Does it match my other layers?** — they must share a CRS before any spatial operation
4. **Is it appropriate for my analysis?** — distance/area measurements require a projected CRS in appropriate units

### Common EPSG codes for Pacific Northwest work

| EPSG | Name | Type | Units | Use case |
|---|---|---|---|---|
| 4326 | WGS 84 | Geographic | Degrees | GPS, web data, GeoJSON |
| 3857 | Web Mercator | Projected | Meters | Web tile basemaps (not for measurement) |
| 2927 | WA State Plane South (ft) | Projected | US survey feet | WA state/county data |
| **32610** | **UTM Zone 10N (WGS 84)** | **Projected** | **Meters** | **Course standard — distance/area measurement in W WA** |
| 4269 | NAD 83 | Geographic | Degrees | USGS and federal data |

> ### ⚠️ The `set_crs` vs. `to_crs` trap — the #1 silent CRS bug
> This is the single most common error in this module. Confusing these two functions produces geometry that looks fine on a map but is silently wrong for every spatial operation.
>
> - **`set_crs()` / `st_set_crs()`** = "I'm declaring what the CRS already is." Use only when the CRS is missing or mislabeled in the metadata. **Does not move coordinates.**
> - **`to_crs()` / `st_transform()`** = "Convert the coordinates to a new CRS." Use to align layers before any spatial operation. **Moves coordinates.**
>
> Mnemonic: `set_crs` = slap a label. `to_crs` / `st_transform` = transform the numbers. When in doubt: check the CRS, then transform.


### 2a — CRS inspection and reprojection in Python

In [ ]:
# [Python] CRS inspection, assignment, and reprojection
import geopandas as gpd
import pandas as pd

# Build a salmon observations GeoDataFrame (synthetic data — no file download needed)
obs_df = pd.DataFrame({
    'obs_id':   ['OBS001', 'OBS002', 'OBS003', 'OBS004', 'OBS005'],
    'species':  ['Chinook', 'Coho', 'Chinook', 'Sockeye', 'Coho'],
    'lon':      [-122.3088, -122.4500, -122.5403, -122.6200, -122.3700],
    'lat':      [  47.6062,   47.5800,   47.6500,   47.7100,   47.5500],
    'count':    [12, 5, 8, 21, 3]
})
obs_4326 = gpd.GeoDataFrame(
    obs_df,
    geometry=gpd.points_from_xy(obs_df['lon'], obs_df['lat']),
    crs='EPSG:4326'
)

# ── Inspect CRS ───────────────────────────────────────────────────────────────
print("CRS:",            obs_4326.crs)
print("EPSG code:",      obs_4326.crs.to_epsg())
print("Is geographic?:", obs_4326.crs.is_geographic)
print("Is projected?:",  obs_4326.crs.is_projected)

# ── set_crs() — assigns a label only, does NOT move coordinates ──────────────
obs_no_crs = obs_4326.copy()
obs_no_crs.crs = None                         # strip CRS for demo
obs_reassigned = obs_no_crs.set_crs('EPSG:4326')   # assign — coordinates unchanged
print("\nAfter set_crs — same coordinates, label restored:", obs_reassigned.crs)

# ── to_crs() — actually transforms coordinates to the course standard ────────
obs_32610 = obs_4326.to_crs('EPSG:32610')
print("\nAfter to_crs(32610) — coordinates transformed:")
print(obs_32610.geometry.head(2))
print("New CRS:", obs_32610.crs)


### 2b — CRS inspection and reprojection in R

In [ ]:
# [R] CRS inspection, assignment, and reprojection with sf
library(sf)

obs_df <- data.frame(
    obs_id  = c('OBS001', 'OBS002', 'OBS003', 'OBS004', 'OBS005'),
    species = c('Chinook', 'Coho', 'Chinook', 'Sockeye', 'Coho'),
    lon     = c(-122.3088, -122.4500, -122.5403, -122.6200, -122.3700),
    lat     = c(  47.6062,   47.5800,   47.6500,   47.7100,   47.5500),
    count   = c(12, 5, 8, 21, 3)
)
obs_4326 <- st_as_sf(obs_df, coords = c('lon', 'lat'), crs = 4326)

# ── Inspect CRS ───────────────────────────────────────────────────────────────
cat("CRS name:     ", st_crs(obs_4326)$Name, "\n")
cat("EPSG code:    ", st_crs(obs_4326)$epsg, "\n")
cat("Is geographic:", st_is_longlat(obs_4326), "\n")

# ── st_set_crs() — assigns a label only, does NOT move coordinates ───────────
obs_no_crs <- obs_4326
st_crs(obs_no_crs) <- NA                       # strip CRS for demo
obs_reassigned <- st_set_crs(obs_no_crs, 4326)  # assign — coordinates unchanged
cat("\nAfter st_set_crs — same coordinates, label restored:", st_crs(obs_reassigned)$epsg, "\n")

# ── st_transform() — actually transforms coordinates to the course standard ──
obs_32610 <- st_transform(obs_4326, crs = 32610)
cat("\nAfter st_transform(32610) — coordinates transformed:\n")
print(head(st_coordinates(obs_32610), 2))
cat("New CRS:", st_crs(obs_32610)$epsg, "\n")


### 2c — Defensive CRS check before any multi-layer workflow

In [ ]:
# [Python] Defensive CRS check function — use at the top of any multi-layer workflow
import geopandas as gpd
import pandas as pd

def ensure_same_crs(*gdfs, target_epsg=32610):
    """
    Verify all GeoDataFrames share a CRS.
    If they differ, reproject all to target_epsg.
    Returns a tuple of aligned GeoDataFrames.
    """
    crs_values = [gdf.crs.to_epsg() if gdf.crs else None for gdf in gdfs]
    unique_crs = set(crs_values)

    if len(unique_crs) == 1 and None not in unique_crs:
        print(f"✅  All layers share CRS EPSG:{list(unique_crs)[0]}")
        return gdfs
    else:
        print(f"⚠️  CRS mismatch detected: {crs_values}")
        print(f"    Reprojecting all to EPSG:{target_epsg}...")
        reprojected = tuple(gdf.to_crs(epsg=target_epsg) for gdf in gdfs)
        print(f"✅  All layers now in EPSG:{target_epsg}")
        return reprojected

# --- Demo: simulate a CRS mismatch between two layers ---------------------
demo_df = pd.DataFrame({'obs_id': ['OBS001', 'OBS002'], 'lon': [-122.31, -122.45], 'lat': [47.61, 47.58]})
layer_a = gpd.GeoDataFrame(demo_df, geometry=gpd.points_from_xy(demo_df.lon, demo_df.lat), crs='EPSG:4326')
layer_b = layer_a.to_crs(epsg=2927)  # simulate a layer that arrived in a different CRS

layer_a_aligned, layer_b_aligned = ensure_same_crs(layer_a, layer_b, target_epsg=32610)
print("Layer A CRS:", layer_a_aligned.crs.to_epsg())
print("Layer B CRS:", layer_b_aligned.crs.to_epsg())


---

## Section 3 — Clean and Standardize Attribute Tables

Run this sequence on every new dataset before any spatial operation:

1. **Standardize column names** — lowercase, strip whitespace, replace spaces/punctuation
2. **Fix data types** — coerce numeric and date columns; let unparseable values become null rather than crashing
3. **Drop rows missing a key join field** — and report how many rows you kept

`errors='coerce'` in pandas and `as.numeric()` / `as.Date()` in R both turn unparseable values into `NA` rather than raising an error — this lets the pipeline continue, so you can inspect what failed afterward instead of debugging a crash mid-script.


### 3a — Attribute cleaning in Python

In [ ]:
# [Python] Attribute table cleaning — same logic as ArcGIS Pro's "Calculate Field" and "Select by Attribute",
# but reproducible, batch-able, and auditable.
import pandas as pd
import geopandas as gpd

# Simulate a messy watershed table — inconsistent casing, stray whitespace, mixed types, some nulls
messy = pd.DataFrame({
    ' Watershed ID ':   ['WS-01', 'WS-02', 'WS-03', None, 'WS-05'],
    'Watershed Name':   ['Cedar River', 'Issaquah Creek', 'Bear Creek', 'Unnamed', 'Soos Creek'],
    'Area_SqFt':        ['4521000', '2110500', 'bad_value', '8870000', '3340000'],
    'Survey Date':      ['2023-04-01', '2023-04-15', '2023/05/02', None, '2023-05-20'],
})

# 1. Column name standardization
messy.columns = (
    messy.columns.str.lower()
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)
print("Standardized columns:", list(messy.columns))

# 2. Data type fixes — unparseable values become NaN, not a crash
messy["area_sqft"]   = pd.to_numeric(messy["area_sqft"], errors="coerce")
messy["survey_date"] = pd.to_datetime(messy["survey_date"], errors="coerce")
print("\nAfter type coercion:")
print(messy[["area_sqft", "survey_date"]])

# 3. Drop rows missing the key join field
clean = messy.dropna(subset=["watershed_id", "area_sqft"])
print(f"\n{len(clean)}/{len(messy)} rows kept after dropping nulls in key fields")
print(clean)


### 3b — Attribute cleaning in R

In [ ]:
# [R] Attribute table cleaning with dplyr + janitor
library(dplyr)
library(janitor)

messy <- data.frame(
    `Watershed ID`   = c('WS-01', 'WS-02', 'WS-03', NA, 'WS-05'),
    `Watershed Name` = c('Cedar River', 'Issaquah Creek', 'Bear Creek', 'Unnamed', 'Soos Creek'),
    Area_SqFt        = c('4521000', '2110500', 'bad_value', '8870000', '3340000'),
    `Survey Date`    = c('2023-04-01', '2023-04-15', '2023-05-02', NA, '2023-05-20'),
    check.names = FALSE
)

# 1. Column name standardization — one call replaces the whole chain
messy <- messy |> clean_names()
print(names(messy))

# 2. Data type fixes — unparseable values become NA, not an error
messy <- messy |>
    mutate(
        area_sq_ft  = as.numeric(area_sq_ft),
        survey_date = as.Date(survey_date, "%Y-%m-%d")
    )
print(messy[c("area_sq_ft", "survey_date")])

# 3. Drop rows missing the key join field
clean <- messy |> filter(!is.na(watershed_id), !is.na(area_sq_ft))
cat(sprintf("\n%d/%d rows kept after dropping nulls in key fields\n", nrow(clean), nrow(messy)))
print(clean)


---

## Section 4 — Tabular → Spatial

Any CSV with coordinate columns can become a spatial dataset in three lines of code. The critical detail: **longitude is X (first argument), latitude is Y (second argument)** — swap them and your points land in the ocean, or worse, somewhere that still looks plausible until someone checks the map.

> ### ⚠️ Watch for lon/lat swap
> `points_from_xy(x=LON, y=LAT)` in Python and `coords = c("longitude", "latitude")` in R both expect **longitude first**. Get the order backwards and you'll generate points in the wrong hemisphere — sometimes in the ocean, which is at least obviously wrong; sometimes just shifted, which is much harder to catch without a plot.
>
> **The plot check is mandatory, not optional.** It catches a swapped lon/lat in about two seconds — far faster than debugging a downstream spatial join that silently returns zero matches.


### 4a — Promote a CSV to spatial in Python

In [ ]:
# [Python] Tabular → Spatial — promote a salmon observation CSV to a GeoDataFrame
import pandas as pd
import geopandas as gpd

# Simulate the salmon observation CSV (in production: pd.read_csv('data/salmon_obs.csv'))
obs_csv = pd.DataFrame({
    'obs_id':  ['OBS101', 'OBS102', 'OBS103', 'OBS104'],
    'species': ['Chinook', 'Coho', 'Sockeye', 'Chinook'],
    'lon':     [-122.3088, -122.4500, -122.5403, -122.2900],
    'lat':     [  47.6062,   47.5800,   47.6500,   47.5200],
    'count':   [14, 6, 22, 9]
})

# CRITICAL: points_from_xy(x=LON, y=LAT) — longitude first, latitude second
obs_gdf = gpd.GeoDataFrame(
    obs_csv,
    geometry=gpd.points_from_xy(
        obs_csv["lon"],   # x = longitude
        obs_csv["lat"]),  # y = latitude
    crs="EPSG:4326"
)

# ── Mandatory sanity checks ───────────────────────────────────────────────────
print("Is valid:",        obs_gdf.geometry.is_valid.all())   # True
print("Row count match:", len(obs_gdf) == len(obs_csv))      # True
print("Geometry type:",   obs_gdf.geom_type.unique())

# ── Mandatory plot check — catches a swapped lon/lat instantly ───────────────
obs_gdf.plot()  # Should show points clustered around Western Washington, not the ocean


### 4b — Promote a CSV to spatial in R

In [ ]:
# [R] Tabular → Spatial — promote a salmon observation CSV to an sf object
library(sf)

obs_csv <- data.frame(
    obs_id  = c('OBS101', 'OBS102', 'OBS103', 'OBS104'),
    species = c('Chinook', 'Coho', 'Sockeye', 'Chinook'),
    lon     = c(-122.3088, -122.4500, -122.5403, -122.2900),
    lat     = c(  47.6062,   47.5800,   47.6500,   47.5200),
    count   = c(14, 6, 22, 9)
)

# CRITICAL: coords = c("longitude", "latitude") — longitude first, latitude second
obs_sf <- st_as_sf(
    obs_csv,
    coords = c("lon", "lat"),
    crs    = 4326
)

# ── Mandatory sanity checks ───────────────────────────────────────────────────
cat("Geometry type:", as.character(unique(st_geometry_type(obs_sf))), "\n")
cat("Row count match:", nrow(obs_sf) == nrow(obs_csv), "\n")   # TRUE

# ── Mandatory plot check — catches a swapped lon/lat instantly ───────────────
plot(st_geometry(obs_sf))  # Should show points clustered around Western Washington, not the ocean


---

## Section 5 — Harmonize and Export

The deliverable for tonight's intake routine is a single, multi-layer GeoPackage: all layers in a common CRS, clean attributes, openable in ArcGIS Pro, R, and Python without any format conversion. GeoPackage is the modern default output — one file, no 10-character field name limit (unlike Shapefile), and readable everywhere.


### 5a — Export a harmonized, multi-layer GeoPackage in Python

In [ ]:
# [Python] Export a clean, multi-layer GeoPackage — the preferred delivery format
import geopandas as gpd
import pandas as pd
from shapely.geometry import box
import os

os.makedirs('output', exist_ok=True)

# ── Layer 1: salmon observations (built in Section 4, reprojected to course standard) ──
obs_csv = pd.DataFrame({
    'obs_id':  ['OBS101', 'OBS102', 'OBS103', 'OBS104'],
    'species': ['Chinook', 'Coho', 'Sockeye', 'Chinook'],
    'lon':     [-122.3088, -122.4500, -122.5403, -122.2900],
    'lat':     [  47.6062,   47.5800,   47.6500,   47.5200],
    'count':   [14, 6, 22, 9]
})
obs_gdf = gpd.GeoDataFrame(
    obs_csv,
    geometry=gpd.points_from_xy(obs_csv["lon"], obs_csv["lat"]),
    crs="EPSG:4326"
).to_crs(epsg=32610)

# ── Layer 2: watershed polygons (synthetic stand-in for a GDB/GeoPackage layer) ──
watersheds = gpd.GeoDataFrame({
    'watershed_id':   ['WS-01', 'WS-02', 'WS-03'],
    'watershed_name': ['Cedar River', 'Issaquah Creek', 'Bear Creek'],
    'geometry': [
        box(-122.35, 47.58, -122.28, 47.64),
        box(-122.48, 47.55, -122.40, 47.61),
        box(-122.58, 47.62, -122.50, 47.68),
    ]
}, crs='EPSG:4326').to_crs(epsg=32610)

# ── Confirm both layers share a CRS before writing ────────────────────────────
assert obs_gdf.crs.to_epsg() == watersheds.crs.to_epsg() == 32610
print("✅  Both layers confirmed in EPSG:32610")

# ── Write the harmonized GeoPackage — one file, two layers ────────────────────
output_path = 'output/salmon_watersheds_harmonized.gpkg'
obs_gdf.to_file(output_path, driver='GPKG', layer='salmon_observations')
watersheds.to_file(output_path, driver='GPKG', layer='watersheds', mode='a')

import fiona
print(f"\n✅  Wrote {output_path}")
print("Layers:", fiona.listlayers(output_path))


### 5b — Export a harmonized, multi-layer GeoPackage in R

In [ ]:
# [R] Export a clean, multi-layer GeoPackage with sf
library(sf)

dir.create("output", showWarnings = FALSE)

# ── Layer 1: salmon observations, reprojected to the course standard ─────────
obs_csv <- data.frame(
    obs_id  = c('OBS101', 'OBS102', 'OBS103', 'OBS104'),
    species = c('Chinook', 'Coho', 'Sockeye', 'Chinook'),
    lon     = c(-122.3088, -122.4500, -122.5403, -122.2900),
    lat     = c(  47.6062,   47.5800,   47.6500,   47.5200),
    count   = c(14, 6, 22, 9)
)
obs_sf <- st_as_sf(obs_csv, coords = c("lon", "lat"), crs = 4326) |> st_transform(32610)

# ── Layer 2: watershed polygons (synthetic stand-in for a GDB/GeoPackage layer) ──
make_box <- function(xmin, ymin, xmax, ymax) {
    st_polygon(list(rbind(
        c(xmin, ymin), c(xmax, ymin), c(xmax, ymax), c(xmin, ymax), c(xmin, ymin)
    )))
}
watersheds_sf <- st_sf(
    watershed_id   = c('WS-01', 'WS-02', 'WS-03'),
    watershed_name = c('Cedar River', 'Issaquah Creek', 'Bear Creek'),
    geometry = st_sfc(
        make_box(-122.35, 47.58, -122.28, 47.64),
        make_box(-122.48, 47.55, -122.40, 47.61),
        make_box(-122.58, 47.62, -122.50, 47.68),
        crs = 4326
    )
) |> st_transform(32610)

# ── Confirm both layers share a CRS before writing ────────────────────────────
stopifnot(st_crs(obs_sf)$epsg == st_crs(watersheds_sf)$epsg, st_crs(obs_sf)$epsg == 32610)
cat("✅  Both layers confirmed in EPSG:32610\n")

# ── Write the harmonized GeoPackage — one file, two layers ────────────────────
output_path <- "output/salmon_watersheds_harmonized_r.gpkg"
st_write(obs_sf, output_path, layer = "salmon_observations", delete_dsn = TRUE, quiet = TRUE)
st_write(watersheds_sf, output_path, layer = "watersheds", append = TRUE, quiet = TRUE)

cat("\n✅  Wrote", output_path, "\n")
cat("Layers:", paste(st_layers(output_path)$name, collapse = ", "), "\n")


---

## Section 6 — Guided Lab: The Intake Pipeline

**Scenario:** It's the start of the week. Salmon observation data comes in as a CSV. The watershed boundaries your agency maintains live in a File Geodatabase (or, for tonight's synthetic version, a GeoPackage standing in for one). A partner agency has also sent a second observation file as GeoJSON. By the end of the lab, all of it needs to live in one harmonized GeoPackage, in a common CRS, ready to open in ArcGIS Pro.

Work through the five steps below with your breakout group. Each step mirrors the Guided Lab structure from tonight's slide deck and matches the five Learning Objectives at the top of this notebook.

**Using the synthetic dataset** (no file download needed): the salmon observation CSV and the watershed boundaries are both constructed inline below, so the lab runs without any external files. In production, swap the inline construction for `gpd.read_file(...)` / `st_read(...)` against your real GDB, hosted layer, or GeoJSON.

**Goal:** A single GeoPackage in `/outputs/` with:
1. A `watersheds` layer — polygon boundaries
2. A `salmon_observations` layer — point layer of observations, with a `watershed_name` column left for Module 4's spatial join
3. Both layers in **EPSG:32610**
4. Clean, standardized column names and no null values in key fields


**Step 1 — Load & Inspect.** Load the salmon observation CSV. Inspect shape, dtypes, and null counts.

In [ ]:
# [Python] Lab Step 1 — Load & Inspect
import pandas as pd
import geopandas as gpd
import os

os.makedirs('output', exist_ok=True)

# In production: obs_raw = pd.read_csv('data/salmon_obs.csv')
# For the lab: synthetic salmon observation data, deliberately a little messy
obs_raw = pd.DataFrame({
    ' Obs ID ':  ['OBS201', 'OBS202', 'OBS203', 'OBS204', 'OBS205', None],
    'Species':   ['Chinook', 'Coho', 'Chinook', 'Sockeye', 'Coho', 'Chinook'],
    'Longitude': [-122.330, -122.455, -122.310, -122.600, -122.470, -122.290],
    'Latitude':  [  47.610,   47.585,   47.625,   47.700,   47.590,   47.615],
    'Count':     ['14', '6', '22', '9', 'bad', '3'],
})

print("Shape:", obs_raw.shape)
print("\nDtypes:\n", obs_raw.dtypes)
print("\nNull counts:\n", obs_raw.isna().sum())


**Step 2 — CRS Check & Reproject.** Load the watershed polygons. Compare CRS across layers. Reproject all to EPSG:32610. Confirm with an assert.

In [ ]:
# [Python] Lab Step 2 — CRS Check & Reproject
from shapely.geometry import box

# In production: watersheds_raw = gpd.read_file('data/watersheds.gdb', layer='watersheds')
# For the lab: synthetic watershed polygons in WGS84 — a deliberate CRS mismatch vs. the target
watersheds_raw = gpd.GeoDataFrame({
    'watershed_id':   ['WS-01', 'WS-02', 'WS-03'],
    'watershed_name': ['Cedar River', 'Issaquah Creek', 'Bear Creek'],
    'geometry': [
        box(-122.35, 47.58, -122.28, 47.64),
        box(-122.48, 47.55, -122.40, 47.61),
        box(-122.62, 47.68, -122.55, 47.72),
    ]
}, crs='EPSG:4326')

print("Watersheds CRS before reprojection:", watersheds_raw.crs)

watersheds_32610 = watersheds_raw.to_crs(epsg=32610)
print("Watersheds CRS after reprojection: ", watersheds_32610.crs)

assert watersheds_32610.crs.to_epsg() == 32610, "Reprojection failed — check the CRS"
print("✅  Watersheds confirmed in EPSG:32610")


**Step 3 — Clean Attributes.** Standardize column names. Fix the `count` data type. Drop rows with a null key field. Report row counts before/after.

In [ ]:
# [Python] Lab Step 3 — Clean Attributes
# 1. Standardize column names
obs_raw.columns = (
    obs_raw.columns.str.lower()
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)
print("Standardized columns:", list(obs_raw.columns))

# 2. Fix the count data type — unparseable values become NaN, not a crash
obs_raw["count"] = pd.to_numeric(obs_raw["count"], errors="coerce")

# 3. Drop rows missing the key join field (obs_id)
obs_clean = obs_raw.dropna(subset=["obs_id"]).copy()

print(f"\n{len(obs_clean)}/{len(obs_raw)} rows kept after dropping nulls in obs_id")
print(obs_clean)


**Step 4 — Tabular → Spatial.** Promote the cleaned CSV to a GeoDataFrame. Verify the row count matches. Plot to confirm.

In [ ]:
# [Python] Lab Step 4 — Tabular → Spatial
# CRITICAL: points_from_xy(x=LON, y=LAT) — longitude first, latitude second
obs_gdf = gpd.GeoDataFrame(
    obs_clean,
    geometry=gpd.points_from_xy(obs_clean["longitude"], obs_clean["latitude"]),
    crs="EPSG:4326"
).to_crs(epsg=32610)

print("Row count match:", len(obs_gdf) == len(obs_clean))   # True
print("Geometry type:",   obs_gdf.geom_type.unique())
print("CRS:",             obs_gdf.crs.to_epsg())

# Mandatory plot check — catches a swapped lon/lat in about two seconds
obs_gdf.plot()  # Should show points clustered around Western Washington, not the ocean


**Step 5 — Export Harmonized Output.** Write both layers to a single GeoPackage in `/outputs/`. Open it in ArcGIS Pro to confirm.

In [ ]:
# [Python] Lab Step 5 — Export Harmonized Output
import fiona

output_gpkg = 'output/module3_lab_output.gpkg'

watersheds_32610.to_file(output_gpkg, driver='GPKG', layer='watersheds')
obs_gdf.to_file(output_gpkg, driver='GPKG', layer='salmon_observations', mode='a')

print(f"✅  Wrote {output_gpkg}")
print("Layers:", fiona.listlayers(output_gpkg))
print("\nNext step (Module 4): a spatial join will populate watershed_name on each observation —")
print("that's the first spatial operation you'll run on this exact GeoPackage next week.")


**Lab Steps — R version.** The same five steps, in R.

In [ ]:
# [R] Lab Steps 1-5 — The Intake Pipeline, R version
library(sf)
library(dplyr)
library(janitor)

dir.create("output", showWarnings = FALSE)

# ── Step 1: Load & Inspect ────────────────────────────────────────────────────
obs_raw <- data.frame(
    `Obs ID`   = c('OBS201', 'OBS202', 'OBS203', 'OBS204', 'OBS205', NA),
    Species    = c('Chinook', 'Coho', 'Chinook', 'Sockeye', 'Coho', 'Chinook'),
    Longitude  = c(-122.330, -122.455, -122.310, -122.600, -122.470, -122.290),
    Latitude   = c(  47.610,   47.585,   47.625,   47.700,   47.590,   47.615),
    Count      = c('14', '6', '22', '9', 'bad', '3'),
    check.names = FALSE
)
str(obs_raw)
colSums(is.na(obs_raw))

# ── Step 2: CRS Check & Reproject ─────────────────────────────────────────────
make_box <- function(xmin, ymin, xmax, ymax) {
    st_polygon(list(rbind(
        c(xmin, ymin), c(xmax, ymin), c(xmax, ymax), c(xmin, ymax), c(xmin, ymin)
    )))
}
watersheds_raw <- st_sf(
    watershed_id   = c('WS-01', 'WS-02', 'WS-03'),
    watershed_name = c('Cedar River', 'Issaquah Creek', 'Bear Creek'),
    geometry = st_sfc(
        make_box(-122.35, 47.58, -122.28, 47.64),
        make_box(-122.48, 47.55, -122.40, 47.61),
        make_box(-122.62, 47.68, -122.55, 47.72),
        crs = 4326
    )
)
cat("Watersheds CRS before reprojection:", st_crs(watersheds_raw)$epsg, "\n")
watersheds_32610 <- st_transform(watersheds_raw, 32610)
cat("Watersheds CRS after reprojection: ", st_crs(watersheds_32610)$epsg, "\n")
stopifnot(st_crs(watersheds_32610)$epsg == 32610)
cat("✅  Watersheds confirmed in EPSG:32610\n")

# ── Step 3: Clean Attributes ──────────────────────────────────────────────────
obs_raw <- obs_raw |> clean_names()
obs_raw <- obs_raw |> mutate(count = as.numeric(count))
obs_clean <- obs_raw |> filter(!is.na(obs_id))
cat(sprintf("\n%d/%d rows kept after dropping nulls in obs_id\n", nrow(obs_clean), nrow(obs_raw)))

# ── Step 4: Tabular → Spatial ──────────────────────────────────────────────────
obs_sf <- st_as_sf(obs_clean, coords = c("longitude", "latitude"), crs = 4326) |> st_transform(32610)
cat("Row count match:", nrow(obs_sf) == nrow(obs_clean), "\n")
plot(st_geometry(obs_sf))  # mandatory plot check

# ── Step 5: Export Harmonized Output ──────────────────────────────────────────
output_gpkg <- "output/module3_lab_output_r.gpkg"
st_write(watersheds_32610, output_gpkg, layer = "watersheds", delete_dsn = TRUE, quiet = TRUE)
st_write(obs_sf, output_gpkg, layer = "salmon_observations", append = TRUE, quiet = TRUE)
cat("\n✅  Wrote", output_gpkg, "\n")
cat("Layers:", paste(st_layers(output_gpkg)$name, collapse = ", "), "\n")


---

## Section 7 — Extended Application: Choose a Challenge

Pick one option below, or apply the intake routine to your own professional data.

**A — Spatial Join Warm-Up (recommended).** Using the salmon observations and watershed polygons you cleaned in the lab — which watershed does each observation fall in? Assign the watershed name as a new column.
- Python: `gpd.sjoin(obs, ws, how='left', predicate='within')`
- R: `st_join(obs, ws, join = st_within)`

This is a preview of Module 4 — you're not expected to fully master spatial joins tonight, just see how directly the intake pipeline output feeds into next week's operations.

**B — Harmonize Two Datasets (data focus).** You've been given a second salmon observation CSV from a different agency. Column names differ, the CRS differs, and one has null geometries. Merge it with your existing dataset into one clean GeoDataFrame / sf object.
- Standardize columns → reproject → drop nulls → `pd.concat()` / `rbind()`

**C — Build a Reusable Function (advanced).** Write `load_and_clean(csv_path, lon_col, lat_col, gpkg_path)` that runs all five lab steps and returns a tuple `(obs_gdf, watershed_gdf)`, both in **EPSG:32610**. Add a docstring, type hints, and a validation step that raises a `ValueError` if the CRS check fails.


In [ ]:
# [Python] Option A — Spatial Join Warm-Up
# This previews Module 4. You are not expected to master predicates tonight —
# just see how directly tonight's GeoPackage feeds next week's first operation.

joined = gpd.sjoin(obs_gdf, watersheds_32610[['watershed_id', 'watershed_name', 'geometry']],
                    how='left', predicate='within')
print(joined[['obs_id', 'species', 'watershed_name']])


In [ ]:
# [Python] Option C — Build a Reusable Function
import geopandas as gpd
import pandas as pd
from pathlib import Path

def load_and_clean(csv_path: str, lon_col: str, lat_col: str, gpkg_path: str,
                    target_epsg: int = 32610) -> tuple:
    """
    Run the full Module 3 intake pipeline on a salmon observation CSV.

    Parameters
    ----------
    csv_path    : path to the salmon observation CSV
    lon_col     : name of the longitude column in the raw CSV
    lat_col     : name of the latitude column in the raw CSV
    gpkg_path   : path to write the harmonized GeoPackage
    target_epsg : target CRS for all output layers (course standard: 32610)

    Returns
    -------
    (obs_gdf, watershed_gdf) : both GeoDataFrames, reprojected to target_epsg

    Raises
    ------
    ValueError : if either output layer fails the CRS check after reprojection
    """
    df = pd.read_csv(csv_path)
    df.columns = (df.columns.str.lower().str.strip()
                  .str.replace(" ", "_", regex=False)
                  .str.replace(r"[^a-z0-9_]", "", regex=True))

    gdf = gpd.GeoDataFrame(
        df, geometry=gpd.points_from_xy(df[lon_col.lower()], df[lat_col.lower()]),
        crs="EPSG:4326"
    ).to_crs(epsg=target_epsg)

    if gdf.crs.to_epsg() != target_epsg:
        raise ValueError(f"CRS check failed: expected EPSG:{target_epsg}, got {gdf.crs.to_epsg()}")

    Path(gpkg_path).parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(gpkg_path, driver='GPKG', layer='salmon_observations')

    return gdf, None  # watershed_gdf omitted in this minimal demo — extend as needed

print("load_and_clean() defined. Call it with a real CSV path to run the full pipeline end to end.")


---

## Section 8 — Debrief: Check Your Understanding

Answer these in your notebook or discussion post before moving to Module 4.

1. When would you reach for SeDF versus GeoPandas — and why? *(Think about: data source — AGOL vs. file, license requirements, downstream publishing needs.)*
2. What is the difference between `set_crs` and `to_crs`? Give an example of when you'd use each. *(One assigns a label. One transforms coordinates. Getting these confused is the most common CRS bug.)*
3. Walk through turning a CSV with lat/lon into a GeoPackage layer ready for ArcGIS Pro. *(Five steps: `read_csv` → `points_from_xy` → set CRS → reproject → `to_file`. Each step has a check.)*
4. What question about tonight's content are you still uncertain about? *(Post to Ed Discussion with `#module3`. Your classmates and instructor will respond.)*

### Common errors — what to watch for

| Error | Why it happens | Fix |
|---|---|---|
| CRS mismatch | Operations silently fail or return empty results when layers have different CRS | Print the CRS for every layer at the top of your script, before any spatial operation |
| `set_crs` vs `to_crs` confusion | `set_crs` looks like it "fixes" a CRS problem but never moves coordinates | Use `set_crs` only when the CRS is missing; use `to_crs` to align layers |
| Lon/lat swap | `points_from_xy(x=LON, y=LAT)` expects longitude first; swapping silently produces wrong coordinates | The plot check is mandatory — it catches a swap in about two seconds |
| Row multiplication in `sjoin` | If a point falls inside multiple polygons, `gpd.sjoin()` returns one row per match | Use `groupby().first()` to deduplicate, or `predicate='within'` to enforce one match (relevant going into Module 4) |

---

## What's Next

**Module 4 — Vector Operations.** Buffers, clip/overlay, and spatial joins on the exact data you cleaned and reprojected tonight. The `module3_lab_output.gpkg` you wrote in Section 6 is the starting point for next week's session.

→ **Before next class:** complete this notebook and confirm your GeoPackage output opens in ArcGIS Pro.

**Module 5 — Raster Data.** Shifting from vectors to grids — data models, arrays, file formats, and loading rasters with `rasterio`, `terra`, and `stars`.

**Modules 9–11 — Full Workflow Pipelines.** Vector + raster combined in automated end-to-end workflows. The intake routine you built tonight becomes the first function in every pipeline.

Questions? Post in Ed Discussion — tag `#module3`.
